# DS_Day01_05 — Banking Customer Transaction Behaviour

This Colab notebook generates **synthetic** customer transactions for six months, validates the data, produces four visualisations and seven insights, flags unusually high activity, and evaluates a KNN model.

**Target:** Predict whether a customer is digital-first in month 6 (UPI + card comprise at least 78% of UPI + card + ATM counts), using only months 1–5. The target definition is an explicit assumption because the challenge does not specify a label. All results describe a simulation, not real bank customers.

## 1. Imports and reproducible setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(5)

## 2. Generate synthetic six-month records

One row represents a customer-month. The month field is needed to assess whether ATM usage changes over time. The generator intentionally makes digital usage rise and ATM usage fall slightly; those assumptions must be disclosed when interpreting results.

In [ ]:
ages = ["18-25", "26-35", "36-50", "51+"]
jobs = ["Student", "Salaried", "Self-employed", "Retired"]
cities = ["Coimbatore", "Chennai", "Bengaluru", "Other"]
rows = []
for i in range(1, 601):
    age, job, city = rng.choice(ages), rng.choice(jobs), rng.choice(cities)
    activity = rng.uniform(.65, 1.6)
    habit = rng.normal(0, 2)
    age_effect = {"18-25":5, "26-35":4, "36-50":2, "51+":0}[age]
    city_effect = {"Bengaluru":2, "Chennai":1, "Coimbatore":0, "Other":-1}[city]
    for month in range(1, 7):
        upi = rng.poisson(max(1, (8 + age_effect + city_effect + habit + month*1.1)*activity))
        card = rng.poisson(max(1, (4 + age_effect*.35 + month*.25)*activity))
        atm = rng.poisson(max(.5, (8 - age_effect*.35 - month*.35)*activity))
        rows.append({
            "Customer_ID": f"C{i:04d}", "Month": month, "Age_Group": age,
            "Occupation": job, "City": city, "UPI_Count": int(upi),
            "Card_Count": int(card), "ATM_Count": int(atm),
            "Deposits": round(rng.gamma(3,6500)*activity,2),
            "Withdrawals": round(rng.gamma(2,3200)*activity,2),
            "Bill_Payments": round(rng.gamma(2,1800)*activity,2)
        })
df = pd.DataFrame(rows)
display(df.head())

## 3. Data validation and descriptive statistics

In [ ]:
counts = ["UPI_Count","Card_Count","ATM_Count"]
amounts = ["Deposits","Withdrawals","Bill_Payments"]
assert df.shape == (3600,11)
assert df.Customer_ID.nunique() == 600
assert df.groupby("Customer_ID").Month.nunique().eq(6).all()
assert not df.duplicated(["Customer_ID","Month"]).any()
assert not df.isna().any().any()
assert df[counts+amounts].ge(0).all().all()
assert df.Month.between(1,6).all()
print("Missing:",df.isna().sum().sum(),"Duplicate customer-month:",df.duplicated(["Customer_ID","Month"]).sum())
display(df[counts+amounts].describe().round(2))
df["Digital_Count"] = df.UPI_Count + df.Card_Count
df["Total_Channel_Count"] = df.Digital_Count + df.ATM_Count
df["Digital_Share"] = np.where(df.Total_Channel_Count > 0, df.Digital_Count/df.Total_Channel_Count, np.nan)
display(df.groupby("Age_Group")[counts+["Digital_Share"]].mean().round(2))

## 4. Four visualisations

In [ ]:
monthly = df.groupby("Month")[counts].mean()
ax = monthly.plot(marker="o", figsize=(9,5))
ax.set(title="1. Average monthly channel transactions", ylabel="Mean count")
plt.tight_layout(); plt.savefig("01_channel_trend.png",dpi=150); plt.show()

plt.figure(figsize=(8,5))
sns.barplot(data=df,x="Age_Group",y="Digital_Share",order=ages,errorbar=None)
plt.title("2. Digital transaction share by age group"); plt.ylim(0,1)
plt.tight_layout(); plt.savefig("02_digital_by_age.png",dpi=150); plt.show()

atm_age = df.groupby(["Month","Age_Group"]).ATM_Count.mean().reset_index()
plt.figure(figsize=(9,5))
sns.lineplot(data=atm_age,x="Month",y="ATM_Count",hue="Age_Group",hue_order=ages,marker="o")
plt.title("3. ATM usage by age group"); plt.ylabel("Mean ATM count")
plt.tight_layout(); plt.savefig("03_atm_by_age.png",dpi=150); plt.show()

plt.figure(figsize=(9,5))
sns.barplot(data=df,x="Occupation",y="Digital_Share",errorbar=None)
plt.title("4. Digital transaction share by occupation"); plt.ylim(0,1)
plt.tight_layout(); plt.savefig("04_digital_by_occupation.png",dpi=150); plt.show()

## 5. Flag unusually high activity

The IQR rule marks customer-months for further review. **High volume is not a fraud label.**

In [ ]:
q1,q3 = df.Total_Channel_Count.quantile([.25,.75])
threshold = q3 + 1.5*(q3-q1)
unusual = df[df.Total_Channel_Count > threshold].sort_values("Total_Channel_Count",ascending=False)
print(f"Review threshold: more than {threshold:.1f} transactions; flagged records: {len(unusual)}")
display(unusual[["Customer_ID","Month","Age_Group","Occupation","UPI_Count","Card_Count","ATM_Count","Total_Channel_Count"]].head(10))

## 6. Prepare customer-level prediction data

The model receives mean transaction behaviour from months 1–5 plus customer attributes. Month-6 transactions determine the target only. Splitting one row per customer keeps customers separate across train and test.

In [ ]:
past = df[df.Month <= 5]
future = df[df.Month == 6].copy()
features = past.groupby("Customer_ID")[counts+amounts+["Digital_Share"]].mean().add_prefix("Past_Mean_")
features = features.join(past.groupby("Customer_ID")[["Age_Group","Occupation","City"]].first())
future["Digital_First_M6"] = future.Digital_Share.ge(.78).astype(int)
model_df = features.join(future.set_index("Customer_ID")["Digital_First_M6"])
display(model_df.Digital_First_M6.value_counts())
display(model_df.head())

## 7. Train and evaluate KNN

KNN uses feature distances. Numeric features are imputed and scaled, while categories are encoded. Grid search tunes neighbours and weights using five training folds; evaluation uses held-out customers.

In [ ]:
X = model_df.drop(columns="Digital_First_M6")
y = model_df.Digital_First_M6
categorical = ["Age_Group","Occupation","City"]
numeric = [c for c in X.columns if c not in categorical]
preprocess = ColumnTransformer([
    ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric),
    ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                    ("encode",OneHotEncoder(handle_unknown="ignore"))]),categorical)
])
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.25,random_state=5,stratify=y)
search = GridSearchCV(
    Pipeline([("preprocess",preprocess),("knn",KNeighborsClassifier())]),
    {"knn__n_neighbors":[3,5,7,9,11,15],"knn__weights":["uniform","distance"]},
    scoring="f1",cv=5,n_jobs=-1
)
search.fit(X_train,y_train)
model = search.best_estimator_
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]
print("Best parameters:",search.best_params_)
print("Train customers:",len(X_train),"Test customers:",len(X_test))
print(classification_report(y_test,pred,zero_division=0))
print("ROC AUC:",round(roc_auc_score(y_test,prob),3))
print("Majority-class baseline accuracy:", round(max(y_test.mean(), 1-y_test.mean()), 3))
print("Confusion matrix:\n",confusion_matrix(y_test,pred))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=["Other","Digital-first"],cmap="Blues")
plt.title("KNN held-out test results"); plt.show()

## 8. Seven calculated insights and interpretation

In [ ]:
age_share = df.groupby("Age_Group").Digital_Share.mean()
job_share = df.groupby("Occupation").Digital_Share.mean()
city_share = df.groupby("City").Digital_Share.mean()
atm_by_age = df[df.Month.isin([1,6])].groupby(["Age_Group","Month"]).ATM_Count.mean().unstack()
largest_drop = (atm_by_age[1]-atm_by_age[6]).idxmax()
insights = [
    f"Mean digital share across the simulated portfolio: {df.Digital_Share.mean():.1%}.",
    f"Mean UPI count increased from {monthly.loc[1,'UPI_Count']:.1f} in month 1 to {monthly.loc[6,'UPI_Count']:.1f} in month 6.",
    f"Mean ATM count changed from {monthly.loc[1,'ATM_Count']:.1f} to {monthly.loc[6,'ATM_Count']:.1f} across six months.",
    f"{age_share.idxmax()} had the highest mean digital share ({age_share.max():.1%}), compared with {age_share.idxmin()} ({age_share.min():.1%}).",
    f"The largest age-group drop in ATM usage was {largest_drop}: {atm_by_age.loc[largest_drop,1]:.1f} to {atm_by_age.loc[largest_drop,6]:.1f}.",
    f"{job_share.idxmax()} led occupations in digital share ({job_share.max():.1%}); {city_share.idxmax()} led cities ({city_share.max():.1%}).",
    f"{len(unusual)} customer-month records exceeded the IQR review threshold ({threshold:.1f} transactions)."
]
for i,x in enumerate(insights,1): print(f"{i}. {x}")
print("\nInterpretation: compare held-out precision, recall, F1 and ROC AUC above.")
print("These patterns and scores reflect the synthetic generator, not real-world banking behaviour.")

## 9. Practical action plan

1. Offer assisted UPI and card onboarding to groups with lower digital share.
2. Monitor monthly ATM use by segment before changing ATM availability.
3. Test customer education campaigns and measure whether adoption changes.
4. Review high-volume records with additional context; do not infer fraud from an IQR flag.
5. Validate with properly governed real customer data before operational use.

**Model interpretation:** The held-out confusion matrix shows that KNN can miss customers in the smaller non-digital-first group. Compare its accuracy to the majority-class baseline and examine both classes before claiming predictive value. The model is a learning exercise, not a deployment candidate.

**Limitations:** Six simulated months cannot establish a sustained real-world ATM decline. The target is a chosen digital preference proxy. Deposits, withdrawals, and bill payments are amounts, not transaction counts. Group differences are associations and do not prove causes.

## 10. Export the generated CSV and four charts

In [ ]:
from pathlib import Path
import shutil
out = Path("banking_outputs")
out.mkdir(exist_ok=True)
df.to_csv(out/"synthetic_banking_transactions.csv",index=False)
for name in ["01_channel_trend.png","02_digital_by_age.png","03_atm_by_age.png","04_digital_by_occupation.png"]:
    shutil.copy(name,out/name)
archive = shutil.make_archive("banking_outputs","zip",root_dir=".",base_dir="banking_outputs")
print("Saved",archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass